# 04 - Demoiselles prototyping

Build and check the face contours, seed points, bounded Voronoi
tessellation, face clipping, and the gender/decade assignment and final
render here, cell by cell, before any of it moves into `src/`. Nothing in
this notebook is wired into the rest of the project until Task 5
graduates the validated code.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import plotly.graph_objects as go
from shapely.geometry import Point, Polygon

from src import config

palette = config.PALETTES["demoiselles"]

## Face contours

Rough polygons digitized from `images/les_demoiselles_davignon.png`, one
per figure. These are never subdivided by seed points and never carry
data — they render as flat decorative fills.

In [2]:
FACE_CONTOURS = [
    [(0.20, 0.70), (0.25, 0.70), (0.30, 0.80), (0.25, 0.90), (0.20, 0.85), (0.15, 0.80)],
    [(0.35, 0.65), (0.45, 0.65), (0.45, 0.80), (0.40, 0.85), (0.35, 0.80), (0.30, 0.70)],
    [(0.50, 0.75), (0.55, 0.70), (0.60, 0.80), (0.60, 0.90), (0.50, 0.95), (0.45, 0.85)],
    [(0.75, 0.75), (0.85, 0.80), (0.90, 0.90), (0.80, 0.95), (0.75, 0.90), (0.70, 0.80)],
    [(0.75, 0.45), (0.85, 0.45), (0.90, 0.60), (0.85, 0.65), (0.75, 0.65), (0.70, 0.55)],
]

FACE_POLYGONS = [Polygon(points) for points in FACE_CONTOURS]

for polygon in FACE_POLYGONS:
    assert polygon.is_valid
len(FACE_POLYGONS)

5

## Seed points

A jittered grid over the whole canvas (figures and background alike),
with points inside any face contour discarded before tessellation.

In [3]:
def generate_seed_points(n=40, seed=config.RANDOM_STATE):
    rng = np.random.default_rng(seed)
    grid_size = int(np.ceil(np.sqrt(n * 1.5)))
    xs = np.linspace(0.03, 0.97, grid_size)
    ys = np.linspace(0.03, 0.97, grid_size)
    candidates = [(x, y) for x in xs for y in ys]
    jitter = rng.uniform(-0.03, 0.03, size=(len(candidates), 2))
    jittered = [(x + jx, y + jy) for (x, y), (jx, jy) in zip(candidates, jitter)]
    points = [
        (x, y) for x, y in jittered
        if not any(polygon.contains(Point(x, y)) for polygon in FACE_POLYGONS)
    ]
    return points[:n]

SEED_POINTS = generate_seed_points()
len(SEED_POINTS)

40

In [4]:
import base64

with open(config.IMAGES_DIR / "les_demoiselles_davignon.png", "rb") as f:
    encoded_image = base64.b64encode(f.read()).decode()

def contour_trace(points, color):
    closed = list(points) + [points[0]]
    xs, ys = zip(*closed)
    return go.Scatter(
        x=list(xs), y=list(ys), mode="lines",
        line=dict(color=color, width=2), fill="none",
        hoverinfo="skip", showlegend=False,
    )

def vertex_labels_trace(face_index, points, color):
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    labels = [f"F{face_index}.{point_index}" for point_index in range(len(points))]
    return go.Scatter(
        x=xs, y=ys, mode="markers+text",
        marker=dict(size=6, color=color),
        text=labels, textposition="top center",
        textfont=dict(color=color, size=10),
        hoverinfo="skip", showlegend=False,
    )

fig = go.Figure()
for face_index, contour in enumerate(FACE_CONTOURS):
    fig.add_trace(contour_trace(contour, "#FF00FF"))
    fig.add_trace(vertex_labels_trace(face_index, contour, "#FF00FF"))
fig.add_trace(go.Scatter(
    x=[p[0] for p in SEED_POINTS], y=[p[1] for p in SEED_POINTS],
    mode="markers", marker=dict(size=5, color="#00FFFF"),
    showlegend=False,
))
fig.add_layout_image(
    dict(
        source=f"data:image/png;base64,{encoded_image}",
        xref="x", yref="y",
        x=0, y=1, sizex=1, sizey=1,
        xanchor="left", yanchor="top",
        sizing="stretch", layer="below",
    )
)
fig.update_layout(
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(visible=False, range=[0, 1], scaleanchor="x"),
    showlegend=False,
    margin=dict(t=20, l=0, r=0, b=0),
)
fig.show()